In [1]:
import pandas as pd
import requests

URL = "https://previous-runs-api.open-meteo.com/v1/forecast"
VARS = ["wind_speed_100m", "shortwave_radiation", "cloud_cover", "temperature_2m"]


def fetch_day1(lat, lon, start, end, model=None):
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start, "end_date": end,
        "hourly": ",".join(f"{v}_previous_day1" for v in VARS),
        "timezone": "UTC", "wind_speed_unit": "ms",
    }
    if model:
        params["models"] = model
    r = requests.get(URL, params=params, timeout=120)
    print("Status:", r.status_code)
    if r.status_code != 200:
        print(r.text[:300])
        r.raise_for_status()
    df = pd.DataFrame(r.json()["hourly"])
    df["time"] = pd.to_datetime(df["time"], utc=True)
    return df


week = fetch_day1(54.0, 1.0, "2025-03-01", "2025-03-07")
print(week.shape)
print(week.isna().sum())
week.head()

Status: 200
(168, 5)
time                                 0
wind_speed_100m_previous_day1        0
shortwave_radiation_previous_day1    0
cloud_cover_previous_day1            0
temperature_2m_previous_day1         0
dtype: int64


,time,wind_speed_100m_previous_day1,shortwave_radiation_previous_day1,cloud_cover_previous_day1,temperature_2m_previous_day1
0,2025-03-01 00:00:00+00:00,3.4,0.0,9,6.2
1,2025-03-01 01:00:00+00:00,4.1,0.0,10,6.1
2,2025-03-01 02:00:00+00:00,4.3,0.0,13,6.2
3,2025-03-01 03:00:00+00:00,4.2,0.0,13,6.1
4,2025-03-01 04:00:00+00:00,4.6,0.0,14,6.0


In [2]:
year = fetch_day1(54.0, 1.0, "2025-10-01", "2026-10-10")
print("Rows:", len(year), "| expected about", 375 * 24)

# Share of missing values per month: anything above 0 is a gap
year.set_index("time").isna().resample("MS").mean().round(3)

Status: 200
Rows: 9000 | expected about 9000


,wind_speed_100m_previous_day1,shortwave_radiation_previous_day1,cloud_cover_previous_day1,temperature_2m_previous_day1
time,,,,
2025-10-01 00:00:00+00:00,0.0,0.0,0.0,0.0
2025-11-01 00:00:00+00:00,0.0,0.0,0.0,0.0
2025-12-01 00:00:00+00:00,0.0,0.0,0.0,0.0
2026-01-01 00:00:00+00:00,0.0,0.0,0.0,0.0
2026-02-01 00:00:00+00:00,0.0,0.0,0.0,0.0
2026-03-01 00:00:00+00:00,0.0,0.0,0.0,0.0
2026-04-01 00:00:00+00:00,0.0,0.0,0.0,0.0
2026-05-01 00:00:00+00:00,0.0,0.0,0.0,0.0
2026-06-01 00:00:00+00:00,0.0,0.0,0.0,0.0


In [3]:
observed = pd.read_parquet("../data/processed/weather_hourly.parquet")[["time", "wind_speed_100m__north_sea"]]
m = year.merge(observed, on="time", how="inner").dropna()

err = m["wind_speed_100m_previous_day1"] - m["wind_speed_100m__north_sea"]
print(f"Hours compared: {len(m):,}")
print(f"Day-ahead wind forecast vs observed: MAE {err.abs().mean():.2f} m/s, bias {err.mean():+.2f} m/s")
print(f"Correlation: {m['wind_speed_100m_previous_day1'].corr(m['wind_speed_100m__north_sea']):.3f}")

Hours compared: 9,000
Day-ahead wind forecast vs observed: MAE 1.44 m/s, bias +0.45 m/s
Correlation: 0.911


In [4]:
from pathlib import Path

files = sorted(Path("../data/raw/weather_forecast_d1").glob("*.parquet"))
fc = pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)

missing = fc[fc[VARS].isna().any(axis=1)]
print("Rows with any missing value, by location:")
print(missing.groupby("location")["time"].agg(["min", "max", "count"]), "\n")

fc["month"] = fc["time"].dt.strftime("%Y-%m")
share = fc.groupby("month")[VARS].apply(lambda g: g.isna().mean()).round(3)
print("Share missing per month (months with any gap only):")
share[share.sum(axis=1) > 0]

Rows with any missing value, by location:
                                          min                       max  count
location                                                                      
east_anglia         2024-01-01 00:00:00+00:00 2024-02-16 08:00:00+00:00   1113
irish_sea           2024-01-01 00:00:00+00:00 2024-03-06 17:00:00+00:00   1578
london              2024-01-01 00:00:00+00:00 2024-03-06 17:00:00+00:00   1578
moray_firth         2024-01-01 00:00:00+00:00 2024-03-06 17:00:00+00:00   1578
north_sea           2024-01-01 00:00:00+00:00 2024-03-06 17:00:00+00:00   1578
south_england_solar 2024-01-01 00:00:00+00:00 2024-03-06 17:00:00+00:00   1578 

Share missing per month (months with any gap only):


,wind_speed_100m,shortwave_radiation,cloud_cover,temperature_2m
month,,,,
2024-01,1.000,0.933,0.933,0.933
2024-02,0.922,0.833,0.086,0.086
2024-03,0.155,0.153,0.000,0.000


**Takeaway:** The Previous Runs API covers the full test year (Oct 2025 to Oct 2026) with zero gaps, and a full year can be fetched in one request. Day-ahead (day 1) 100m wind forecasts at the North Sea point vs observed (ERA5): correlation 0.91, MAE 1.44 m/s, bias +0.45 m/s (forecasts windier). Expect honest backtest scores to be worse than with observed weather, and the bias may push predictions low; training on forecast weather is a candidate fix.

**Coverage finding:** Day-ahead forecasts have gaps only at the start of the archive: wind and radiation until early March 2024, cloud and temperature until February 2024. Last missing hour: 2024-03-06 17:00 UTC, identical across locations (East Anglia earlier). From 2024-03-07 every variable and location is complete. The processed forecast table starts there; raw files are left untouched. Tuning (Oct 2024+) and test (Oct 2025+) years are unaffected.